# Final project: Time-series data and application to stock markets {-}

This project aims at familiarizing you with time-series data analysis and its application to stock markets. Datasets you will be working on are Nasdaq and Vietnam stock datasets.

### Submission {-}
The structure of submission folder should be organized as follows:

- ./\<StudentID>-project-notebook.ipynb: Jupyter notebook containing source code.
- ./\<StudentID>-project-report.pdf: project report.

The submission folder is named DL4AI-\<StudentID>-project (e.g., DL4AI-2012345-project) and then compressed with the same name.
    
### Evaluation {-}
Project evaluation will be conducted on how you accomplish the assignment requirements. You can refer to the project instruction slide deck for details.

### Deadline {-}
Please visit Canvas for details.

# Task 1

## Task 1.1

### Load Module & Preprocess Data

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, LSTM
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.callbacks import EarlyStopping

# The nasdaq-100 companies list in technology provided by wikipedia
nasdaq_100_technology = [
    "ADBE", "AMD", "GOOGL", "GOOG", "ADI", "AAPL", "AMAT", "APP", "ARM", 
    "ASML", "ADSK", "AVGO", "CDNS", "CTSH", "CRWD", "DDOG", "DASH", "FTNT", 
    "INTC", "INTU", "KLAC", "LRCX", "MRVL", "META", "MCHP", "MU", "MSFT", 
    "MSTR", "MPWR", "NVDA", "NXPI", "PLTR", "PANW", "PDD", "QCOM", "ROP", 
    "SNDK", "STX", "SHOP", "SNPS", "TXN", "TRI", "WDC", "WDAY", "ZS"
]

# Dataset folder path in Kaggle
folder_path = '/kaggle/input/datasets/pnk1926/final-project-dataset/data_nasdaq_csv/csv' 
cleaned_companies_dict = {}

for company in nasdaq_100_technology:
    file_path = os.path.join(folder_path, f"{company}.csv")
    
    if os.path.exists(file_path):
        # Load CSV data
        df = pd.read_csv(file_path, on_bad_lines='skip')

        # Check for valid number of trading days
        if len(df) < 120: 
            continue
        
        # Check for missing values
        if df.isna().values.any():
            # Apply linear interpolation to fill NaNs without breaking time continuity
            df = df.interpolate(method='linear', limit_direction='both', numeric_only=True)

        # Store the cleaned DataFrame
        cleaned_companies_dict[company] = df

2026-05-11 06:20:22.774031: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778480423.143986      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778480423.243720      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778480424.191722      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778480424.191766      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778480424.191769      23 computation_placer.cc:177] computation placer alr

In [2]:
# Define window and forecasting parameters
window_size = 30 
num_features = 6
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Split the dataset into time windows to get data samples for each company
for company, df in cleaned_companies_dict.items():
    # Extract 6 features (Low, Open, Volume, High, Close, Adj Close)
    values = df.iloc[:, 1:7].values.astype(np.float32)
        
    # Create sliding windows for X (Input features)
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)

    # Create target label y (the next day)
    label_start_idx = window_size
    y_company_final = values[label_start_idx:, 1] 
    
    # Align X and y
    num_valid_samples = len(values) - label_start_idx
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = y_company_final.reshape(-1, 1)

    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Print shape of the training, validation and test set
print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (134738, 30, 6)
Shape of validation set:  (33699, 30, 6)
Shape of test set:  (42121, 30, 6)


In [3]:
# MinMax normalization
def normalize_windows(X, y, target_idx=1):
    X_norm = X.copy()
    y_norm = y.copy()
    ranges_list = []
    mins_list = []
    for i in range(len(X)):
        # Calculate local min and max for the current window
        min_f = np.min(X[i], axis=0)
        max_f = np.max(X[i], axis=0)
        range_f = max_f - min_f
        
        # Prevent division by zero
        range_f[range_f == 0] = 1.0
        
        # Normalize features in X
        X_norm[i] = (X[i] - min_f) / range_f
        
        # Normalize the target y
        y_norm[i] = (y[i] - min_f[target_idx]) / range_f[target_idx]
        
        # Save range and min of the target column for this window
        ranges_list.append(range_f[target_idx])
        mins_list.append(min_f[target_idx])

    return X_norm, y_norm, np.array(ranges_list).reshape(-1, 1), np.array(mins_list).reshape(-1, 1)

X_train_norm, y_train_norm, _, _ = normalize_windows(X_train, y_train, target_idx=1)
X_val_norm, y_val_norm, _, _ = normalize_windows(X_val, y_val, target_idx=1)
X_test_norm, y_test_norm, test_ranges, test_mins = normalize_windows(X_test, y_test, target_idx=1)

### Build & Train Model 

In [4]:
# Build the model architecture
model_1_1 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_1_1.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_1_1.add(BatchNormalization()) 
model_1_1.add(MaxPooling1D(pool_size=2))
model_1_1.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_1_1.add(LSTM(64, return_sequences=True)) 
model_1_1.add(Dropout(0.2))
model_1_1.add(LSTM(32, return_sequences=False))
model_1_1.add(Dropout(0.2))

# Output Block
model_1_1.add(Dense(32, activation='relu'))
model_1_1.add(Dense(1))

# Compile the model
metrics = [
    'mae'
]
model_1_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='mae', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True 
)

# Train the model
history = model_1_1.fit(
    X_train_norm, y_train_norm, 
    validation_data=(X_val_norm, y_val_norm), 
    epochs=30,
    batch_size=512, 
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1778480467.832582      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778480467.838490      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1778480473.057233      71 cuda_dnn.cc:529] Loaded cuDNN version 91002


264/264 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 0.1678 - mae: 0.1678 - val_loss: 0.2479 - val_mae: 0.2479
Epoch 2/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0743 - mae: 0.0743 - val_loss: 0.1244 - val_mae: 0.1244
Epoch 3/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0666 - mae: 0.0666 - val_loss: 0.1079 - val_mae: 0.1079
Epoch 4/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0641 - mae: 0.0641 - val_loss: 0.0848 - val_mae: 0.0848
Epoch 5/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0611 - mae: 0.0611 - val_loss: 0.1078 - val_mae: 0.1078
Epoch 6/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0600 - mae: 0.0600 - val_loss: 0.0815 - val_mae: 0.0815
Epoch 7/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0587 - mae: 0.0587 - val_loss: 0.1092 - val_mae: 0.1092
Epoch 8/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0585 - mae: 0.0585 - val_loss: 0.1109 - val_mae: 0.1109
Epoch 9/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.056

### Validate Model

In [5]:
# Get prediction on the test data
y_pred_norm = model_1_1.predict(X_test_norm)

# 3. Denormalize to USD
y_pred_real = (y_pred_norm * test_ranges) + test_mins
y_test_real = (y_test_norm * test_ranges) + test_mins

print(f"MAE on the test set: {mean_absolute_error(y_pred_norm, y_test_norm):.6f}")
print(f"Actual Error (MAE): ${mean_absolute_error(y_test_real, y_pred_real):.4f} USD")

1317/1317 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
MAE on the test set: 0.084726
Actual Error (MAE): $1.8055 USD


## Task 1.2

### Preprocess Data

In [6]:
# Define window and forecasting parameters
window_size = 30 
k_day = 3 # the next kth day
num_features = 6
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Split the dataset into time windows to get data samples.
for company, df in cleaned_companies_dict.items():
    # Extract 6 features (Low, Open, Volume, High, Close, Adj Close)
    values = df.iloc[:, 1:7].values.astype(np.float32)
        
    # Create sliding windows for X (Input features)
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)

    # Create target label y (the next kth day)
    label_start_idx = window_size + k_day - 1
    y_company_final = values[label_start_idx:, 1] 
    
    # Align X and y
    num_valid_samples = len(values) - label_start_idx
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = y_company_final.reshape(-1, 1)

    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Print shape of the training, validation and test set
print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (134704, 30, 6)
Shape of validation set:  (33687, 30, 6)
Shape of test set:  (42115, 30, 6)


In [7]:
# MinMax normalization
def normalize_windows(X, y, target_idx=1):
    X_norm = X.copy()
    y_norm = y.copy()
    ranges_list = []
    mins_list = []
    for i in range(len(X)):
        # Calculate local min and max for the current window
        min_f = np.min(X[i], axis=0)
        max_f = np.max(X[i], axis=0)
        range_f = max_f - min_f
        
        # Prevent division by zero
        range_f[range_f == 0] = 1.0
        
        # Normalize features in X
        X_norm[i] = (X[i] - min_f) / range_f
        
        # Normalize the target y
        y_norm[i] = (y[i] - min_f[target_idx]) / range_f[target_idx]
        
        # Save range and min of the target column for this window
        ranges_list.append(range_f[target_idx])
        mins_list.append(min_f[target_idx])

    return X_norm, y_norm, np.array(ranges_list).reshape(-1, 1), np.array(mins_list).reshape(-1, 1)

X_train_norm, y_train_norm, _, _ = normalize_windows(X_train, y_train, target_idx=1)
X_val_norm, y_val_norm, _, _ = normalize_windows(X_val, y_val, target_idx=1)
X_test_norm, y_test_norm, test_ranges, test_mins = normalize_windows(X_test, y_test, target_idx=1)

### Build & Train Model 

In [8]:
# Build the model architecture
model_1_2 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_1_2.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_1_2.add(BatchNormalization()) 
model_1_2.add(MaxPooling1D(pool_size=2))
model_1_2.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_1_2.add(LSTM(64, return_sequences=True)) 
model_1_2.add(Dropout(0.2))
model_1_2.add(LSTM(32, return_sequences=False))
model_1_2.add(Dropout(0.2))

# Output Block
model_1_2.add(Dense(32, activation='relu'))
model_1_2.add(Dense(1))

# Compile the model
metrics = [
    'mae'
]
model_1_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='mae', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True 
)

# Train the model
history = model_1_2.fit(
    X_train_norm, y_train_norm, 
    validation_data=(X_val_norm, y_val_norm), 
    epochs=30,
    batch_size=512, 
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.2397 - mae: 0.2397 - val_loss: 0.2888 - val_mae: 0.2888
Epoch 2/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.1808 - mae: 0.1808 - val_loss: 0.2281 - val_mae: 0.2281
Epoch 3/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1779 - mae: 0.1779 - val_loss: 0.2044 - val_mae: 0.2044
Epoch 4/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1755 - mae: 0.1755 - val_loss: 0.1990 - val_mae: 0.1990
Epoch 5/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1743 - mae: 0.1743 - val_loss: 0.1890 - val_mae: 0.1890
Epoch 6/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.1739 - mae: 0.1739 - val_loss: 0.2001 - val_mae: 0.2001
Epoch 7/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1742 - mae: 0.1742 - val_loss: 0.2001 - val_mae: 0.2001
Epoch 8/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1723 - mae: 0.1723 - val_loss: 0.1903 - val_mae: 0.1903


### Validate Model

In [9]:
# Get prediction on the test data
y_pred_norm = model_1_2.predict(X_test_norm)

# 3. Denormalize to USD
y_pred_real = (y_pred_norm * test_ranges) + test_mins
y_test_real = (y_test_norm * test_ranges) + test_mins

print(f"MAE on the test set: {mean_absolute_error(y_pred_norm, y_test_norm):.6f}")
print(f"Actual Error (MAE): ${mean_absolute_error(y_test_real, y_pred_real):.4f} USD")

1317/1317 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
MAE on the test set: 0.188345
Actual Error (MAE): $3.8553 USD


## Task 1.3

### Preprocess Data

In [10]:
# Define window and forecasting parameters
window_size = 30 
k_days = 3 # the next k consecutive days
num_features = 6
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Split the dataset into time windows to get data samples.
for company, df in cleaned_companies_dict.items():
    # Extract features (Low, Open, Volume, High, Close, Adj Close)
    values = df.iloc[:, 1:7].values.astype(np.float32)
    
    # Create sliding windows for X (Input features)
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)
        
    # Create target label y (the next k consecutive days)
    target_column = values[:, 1]
    y_company_views = sliding_window_view(target_column, k_days)
        
    # Align X and y
    num_valid_samples = len(values) - window_size - k_days + 1
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = y_company_views[window_size : window_size + num_valid_samples]

    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Print shape of the training, validation and test set
print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (134704, 30, 6)
Shape of validation set:  (33687, 30, 6)
Shape of test set:  (42115, 30, 6)


In [11]:
# MinMax normalization
def normalize_windows(X, y, target_idx=1):
    X_norm = X.copy()
    y_norm = y.copy()
    ranges_list = []
    mins_list = []
    for i in range(len(X)):
        # Calculate local min and max for the current window
        min_f = np.min(X[i], axis=0)
        max_f = np.max(X[i], axis=0)
        range_f = max_f - min_f
        
        # Prevent division by zero
        range_f[range_f == 0] = 1.0
        
        # Normalize features in X
        X_norm[i] = (X[i] - min_f) / range_f
        
        # Normalize the target y
        y_norm[i] = (y[i] - min_f[target_idx]) / range_f[target_idx]
        
        # Save range and min of the target column for this window
        ranges_list.append(range_f[target_idx])
        mins_list.append(min_f[target_idx])

    return X_norm, y_norm, np.array(ranges_list).reshape(-1, 1), np.array(mins_list).reshape(-1, 1)

X_train_norm, y_train_norm, _, _ = normalize_windows(X_train, y_train, target_idx=1)
X_val_norm, y_val_norm, _, _ = normalize_windows(X_val, y_val, target_idx=1)
X_test_norm, y_test_norm, test_ranges, test_mins = normalize_windows(X_test, y_test, target_idx=1)

### Build & Train Model 

In [12]:
# Build the model architecture
model_1_3 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_1_3.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_1_3.add(BatchNormalization()) 
model_1_3.add(MaxPooling1D(pool_size=2))
model_1_3.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_1_3.add(LSTM(64, return_sequences=True)) 
model_1_3.add(Dropout(0.2))
model_1_3.add(LSTM(32, return_sequences=False))
model_1_3.add(Dropout(0.2))

# Output Block
model_1_3.add(Dense(32, activation='relu'))
model_1_3.add(Dense(k_days))

# Compile the model
metrics = [
    'mae'
]
model_1_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='mae', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True 
)

# Train the model
history = model_1_3.fit(
    X_train_norm, y_train_norm, 
    validation_data=(X_val_norm, y_val_norm), 
    epochs=30,
    batch_size=512, 
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - loss: 0.2327 - mae: 0.2327 - val_loss: 0.2623 - val_mae: 0.2623
Epoch 2/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1322 - mae: 0.1322 - val_loss: 0.1941 - val_mae: 0.1941
Epoch 3/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1256 - mae: 0.1256 - val_loss: 0.1554 - val_mae: 0.1554
Epoch 4/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1230 - mae: 0.1230 - val_loss: 0.1597 - val_mae: 0.1597
Epoch 5/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.1218 - mae: 0.1218 - val_loss: 0.1474 - val_mae: 0.1474
Epoch 6/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1207 - mae: 0.1207 - val_loss: 0.1394 - val_mae: 0.1394
Epoch 7/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1199 - mae: 0.1199 - val_loss: 0.1570 - val_mae: 0.1570
Epoch 8/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.1191 - mae: 0.1191 - val_loss: 0.1531 - val_mae: 0.1531
Epoch 9/30
264/264 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - l

### Validate Model

In [13]:
# Get prediction on the test data
y_pred_norm = model_1_3.predict(X_test_norm)

# 3. Denormalize to USD
y_pred_real = (y_pred_norm * test_ranges) + test_mins
y_test_real = (y_test_norm * test_ranges) + test_mins

for day in range(k_days):
    day_mae = mean_absolute_error(y_test_norm[:, day], y_pred_norm[:, day])
    day_mae_real = mean_absolute_error(y_test_real[:, day], y_pred_real[:, day])
    print(f"Day {day+1} -> MAE on the test set: {day_mae:.6f} | Actual Error (MAE): ${day_mae_real:.4f} USD")

1317/1317 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Day 1 -> MAE on the test set: 0.085924 | Actual Error (MAE): $1.8194 USD
Day 2 -> MAE on the test set: 0.148657 | Actual Error (MAE): $3.0566 USD
Day 3 -> MAE on the test set: 0.187965 | Actual Error (MAE): $3.8351 USD


# Task 2

## Task 2.1

### Load Module & Preprocess Data

In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization, LSTM
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.callbacks import EarlyStopping

# The Vietnamese companies list in technology
VN_technology = [
    "CKV", "CMG", "CMT", "ICT", "ELC", "FPT", "HIG", "HPT", "ITD", "KST", 
    "LTC", "ONE", "PMJ", "PMT", "POT", "SAM", "SBD", "SGT", "SMT", "SRA", 
    "SRB", "ST8", "TST", "UNI", "VAT", "VEC", "VIE", "VLA", "VTC", "VTE"
]

# Dataset folder path in Kaggle
folder_path = '/kaggle/input/datasets/pnk1926/final-project-dataset/data-vn-20230228/stock-historical-data' 
cleaned_companies_dict = {}

all_files = os.listdir(folder_path)
for company in VN_technology:
    matching_files = [f for f in all_files if f.startswith(f"{company}-")]
    if matching_files:
        actual_file_name = matching_files[0]
        file_path = os.path.join(folder_path, actual_file_name)
        # Load CSV data
        df = pd.read_csv(file_path, on_bad_lines='skip')
        
        # Check for valid number of trading days
        if len(df) < 120: 
            continue
            
        # Check for missing values
        if df.isna().values.any():
            # Apply linear interpolation to fill NaNs without breaking time continuity
            df = df.interpolate(method='linear', limit_direction='both', numeric_only=True)

        avg_price = df['Close'].mean()
        if avg_price < 5000:
            continue

        # Store the cleaned DataFrame
        cleaned_companies_dict[company] = df

/tmp/ipykernel_23/1279818414.py:40: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.interpolate(method='linear', limit_direction='both', numeric_only=True)


In [15]:
# Define window and forecasting parameters
window_size = 30 
num_features = 5
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Split the dataset into time windows to get data samples.
for company, df in cleaned_companies_dict.items():
    # Extract 6 features (Low, Open, Volume, High, Close, Adj Close)
    values = df.iloc[:, 1:6].values.astype(np.float32)
        
    # Create sliding windows for X (Input features)
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)

    # Create target label y (the next day)
    label_start_idx = window_size
    y_company_final = values[label_start_idx:, 0] 
    
    # Align X and y
    num_valid_samples = len(values) - label_start_idx
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = y_company_final.reshape(-1, 1)

    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Print shape of the training, validation and test set
print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (48911, 30, 5)
Shape of validation set:  (12238, 30, 5)
Shape of test set:  (15300, 30, 5)


In [16]:
# MinMax normalization
def normalize_windows(X, y, target_idx=0):
    X_norm = X.copy()
    y_norm = y.copy()
    ranges_list = []
    mins_list = []
    for i in range(len(X)):
        # Calculate local min and max for the current window
        min_f = np.min(X[i], axis=0)
        max_f = np.max(X[i], axis=0)
        range_f = max_f - min_f
        
        # Prevent division by zero
        range_f[range_f == 0] = 1.0
        
        # Normalize features in X
        X_norm[i] = (X[i] - min_f) / range_f
        
        # Normalize the target y
        y_norm[i] = (y[i] - min_f[target_idx]) / range_f[target_idx]
        
        # Save range and min of the target column for this window
        ranges_list.append(range_f[target_idx])
        mins_list.append(min_f[target_idx])

    return X_norm, y_norm, np.array(ranges_list).reshape(-1, 1), np.array(mins_list).reshape(-1, 1)

X_train_norm, y_train_norm, _, _ = normalize_windows(X_train, y_train, target_idx=0)
X_val_norm, y_val_norm, _, _ = normalize_windows(X_val, y_val, target_idx=0)
X_test_norm, y_test_norm, test_ranges, test_mins = normalize_windows(X_test, y_test, target_idx=0)

### Build & Train Model 

In [17]:
# Build the model architecture
model_2_1 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_2_1.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_2_1.add(BatchNormalization()) 
model_2_1.add(MaxPooling1D(pool_size=2))
model_2_1.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_2_1.add(LSTM(64, return_sequences=True)) 
model_2_1.add(Dropout(0.2))
model_2_1.add(LSTM(32, return_sequences=False))
model_2_1.add(Dropout(0.2))

# Output Block
model_2_1.add(Dense(32, activation='relu'))
model_2_1.add(Dense(1))

# Compile the model
metrics = [
    'mae'
]
model_2_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='mae', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True 
)

# Train the model
history = model_2_1.fit(
    X_train_norm, y_train_norm, 
    validation_data=(X_val_norm, y_val_norm), 
    epochs=30,
    batch_size=512, 
    callbacks=[early_stop]
)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


96/96 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 1.2811 - mae: 1.2811 - val_loss: 1.8979 - val_mae: 1.8979
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.0262 - mae: 1.0262 - val_loss: 1.8806 - val_mae: 1.8806
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.0674 - mae: 1.0674 - val_loss: 1.8602 - val_mae: 1.8602
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.2533 - mae: 1.2533 - val_loss: 1.8177 - val_mae: 1.8177
Epoch 5/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.7980 - mae: 0.7980 - val_loss: 1.7917 - val_mae: 1.7917
Epoch 6/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.7522 - mae: 0.7522 - val_loss: 1.7739 - val_mae: 1.7739
Epoch 7/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.6512 - mae: 0.6512 - val_loss: 1.7655 - val_mae: 1.7655
Epoch 8/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.7986 - mae: 0.7986 - val_loss: 1.7551 - val_mae: 1.7551
Epoch 9/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.7972 - mae: 0.

### Validate Model

In [18]:
# Get prediction on the test data
y_pred_norm = model_2_1.predict(X_test_norm)

# 3. Denormalize to VND
y_pred_real = (y_pred_norm * test_ranges) + test_mins
y_test_real = (y_test_norm * test_ranges) + test_mins

print(f"MAE on the test set: {mean_absolute_error(y_pred_norm, y_test_norm):.6f}")
print(f"Actual Error (MAE): ${mean_absolute_error(y_test_real, y_pred_real):.4f} VND")

479/479 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
MAE on the test set: 1.238276
Actual Error (MAE): $369.4511 VND


## Task 2.2

### Load Module & Preprocess Data

In [19]:
# Define window and forecasting parameters
window_size = 30 
k_day = 3 # the next kth day
num_features = 5
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Split the dataset into time windows to get data samples.
for company, df in cleaned_companies_dict.items():
    # Extract 6 features (Low, Open, Volume, High, Close, Adj Close)
    values = df.iloc[:, 1:6].values.astype(np.float32)
        
    # Create sliding windows for X (Input features)
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)

    # Create target label y (the next kth day)
    label_start_idx = window_size + k_day - 1
    y_company_final = values[label_start_idx:, 0] 
    
    # Align X and y
    num_valid_samples = len(values) - label_start_idx
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = y_company_final.reshape(-1, 1)

    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Print shape of the training, validation and test set
print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (48876, 30, 5)
Shape of validation set:  (12234, 30, 5)
Shape of test set:  (15289, 30, 5)


In [20]:
# MinMax normalization
def normalize_windows(X, y, target_idx=0):
    X_norm = X.copy()
    y_norm = y.copy()
    ranges_list = []
    mins_list = []
    for i in range(len(X)):
        # Calculate local min and max for the current window
        min_f = np.min(X[i], axis=0)
        max_f = np.max(X[i], axis=0)
        range_f = max_f - min_f
        
        # Prevent division by zero
        range_f[range_f == 0] = 1.0
        
        # Normalize features in X
        X_norm[i] = (X[i] - min_f) / range_f
        
        # Normalize the target y
        y_norm[i] = (y[i] - min_f[target_idx]) / range_f[target_idx]
        
        # Save range and min of the target column for this window
        ranges_list.append(range_f[target_idx])
        mins_list.append(min_f[target_idx])

    return X_norm, y_norm, np.array(ranges_list).reshape(-1, 1), np.array(mins_list).reshape(-1, 1)

X_train_norm, y_train_norm, _, _ = normalize_windows(X_train, y_train, target_idx=0)
X_val_norm, y_val_norm, _, _ = normalize_windows(X_val, y_val, target_idx=0)
X_test_norm, y_test_norm, test_ranges, test_mins = normalize_windows(X_test, y_test, target_idx=0)

### Build & Train Model 

In [21]:
# Build the model architecture
model_2_2 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_2_2.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_2_2.add(BatchNormalization()) 
model_2_2.add(MaxPooling1D(pool_size=2))
model_2_2.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_2_2.add(LSTM(64, return_sequences=True)) 
model_2_2.add(Dropout(0.2))
model_2_2.add(LSTM(32, return_sequences=False))
model_2_2.add(Dropout(0.2))

# Output Block
model_2_2.add(Dense(32, activation='relu'))
model_2_2.add(Dense(1))

# Compile the model
metrics = [
    'mae'
]
model_2_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='mae', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True 
)

# Train the model
history = model_2_2.fit(
    X_train_norm, y_train_norm, 
    validation_data=(X_val_norm, y_val_norm), 
    epochs=30,
    batch_size=512, 
    callbacks=[early_stop]
)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


96/96 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - loss: 2.8987 - mae: 2.8987 - val_loss: 4.9085 - val_mae: 4.9085
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.7218 - mae: 2.7218 - val_loss: 4.8849 - val_mae: 4.8849
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.1943 - mae: 2.1943 - val_loss: 4.8551 - val_mae: 4.8551
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.9856 - mae: 2.9856 - val_loss: 4.8205 - val_mae: 4.8205
Epoch 5/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.2339 - mae: 2.2339 - val_loss: 4.8015 - val_mae: 4.8015
Epoch 6/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.2261 - mae: 2.2261 - val_loss: 4.7867 - val_mae: 4.7867
Epoch 7/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.1043 - mae: 2.1043 - val_loss: 4.7854 - val_mae: 4.7854
Epoch 8/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.6841 - mae: 2.6841 - val_loss: 4.7879 - val_mae: 4.7879
Epoch 9/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.4448 - mae: 2.

### Validate Model

In [22]:
# Get prediction on the test data
y_pred_norm = model_2_2.predict(X_test_norm)

# 3. Denormalize to VND
y_pred_real = (y_pred_norm * test_ranges) + test_mins
y_test_real = (y_test_norm * test_ranges) + test_mins

print(f"MAE on the test set: {mean_absolute_error(y_pred_norm, y_test_norm):.6f}")
print(f"Actual Error (MAE): ${mean_absolute_error(y_test_real, y_pred_real):.4f} VND")

478/478 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
MAE on the test set: 3.742477
Actual Error (MAE): $657.4317 VND


## Task 2.3

### Preprocess Data

In [23]:
# Define window and forecasting parameters
window_size = 30 
k_days = 3 # the next k consecutive days
num_features = 5
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Split the dataset into time windows to get data samples.
for company, df in cleaned_companies_dict.items():
    # Extract features (Low, Open, Volume, High, Close, Adj Close)
    values = df.iloc[:, 1:6].values.astype(np.float32)
    
    # Create sliding windows for X (Input features)
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)
        
    # Create target label y (the next k consecutive days)
    target_column = values[:, 0]
    y_company_views = sliding_window_view(target_column, k_days)
        
    # Align X and y
    num_valid_samples = len(values) - window_size - k_days + 1
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = y_company_views[window_size : window_size + num_valid_samples]

    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)

X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)

X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Print shape of the training, validation and test set
print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (48876, 30, 5)
Shape of validation set:  (12234, 30, 5)
Shape of test set:  (15289, 30, 5)


In [24]:
# MinMax normalization
def normalize_windows(X, y, target_idx=0):
    X_norm = X.copy()
    y_norm = y.copy()
    ranges_list = []
    mins_list = []
    for i in range(len(X)):
        # Calculate local min and max for the current window
        min_f = np.min(X[i], axis=0)
        max_f = np.max(X[i], axis=0)
        range_f = max_f - min_f
        
        # Prevent division by zero
        range_f[range_f == 0] = 1.0
        
        # Normalize features in X
        X_norm[i] = (X[i] - min_f) / range_f
        
        # Normalize the target y
        y_norm[i] = (y[i] - min_f[target_idx]) / range_f[target_idx]
        
        # Save range and min of the target column for this window
        ranges_list.append(range_f[target_idx])
        mins_list.append(min_f[target_idx])

    return X_norm, y_norm, np.array(ranges_list).reshape(-1, 1), np.array(mins_list).reshape(-1, 1)

X_train_norm, y_train_norm, _, _ = normalize_windows(X_train, y_train, target_idx=0)
X_val_norm, y_val_norm, _, _ = normalize_windows(X_val, y_val, target_idx=0)
X_test_norm, y_test_norm, test_ranges, test_mins = normalize_windows(X_test, y_test, target_idx=0)

### Build & Train Model 

In [25]:
# Build the model architecture
model_2_3 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_2_3.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_2_3.add(BatchNormalization()) 
model_2_3.add(MaxPooling1D(pool_size=2))
model_2_3.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_2_3.add(LSTM(64, return_sequences=True)) 
model_2_3.add(Dropout(0.2))
model_2_3.add(LSTM(32, return_sequences=False))
model_2_3.add(Dropout(0.2))

# Output Block
model_2_3.add(Dense(32, activation='relu'))
model_2_3.add(Dense(k_days))

# Compile the model
metrics = [
    'mae'
]
model_2_3.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='mae', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True 
)

# Train the model
history = model_2_3.fit(
    X_train_norm, y_train_norm, 
    validation_data=(X_val_norm, y_val_norm), 
    epochs=30,
    batch_size=512, 
    callbacks=[early_stop]
)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


96/96 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - loss: 1.9320 - mae: 1.9320 - val_loss: 3.3876 - val_mae: 3.3876
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6206 - mae: 1.6206 - val_loss: 3.3521 - val_mae: 3.3521
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 2.2421 - mae: 2.2421 - val_loss: 3.3305 - val_mae: 3.3305
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.7298 - mae: 1.7298 - val_loss: 3.3041 - val_mae: 3.3041
Epoch 5/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.8550 - mae: 1.8550 - val_loss: 3.2660 - val_mae: 3.2660
Epoch 6/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.6272 - mae: 1.6272 - val_loss: 3.2490 - val_mae: 3.2490
Epoch 7/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.8141 - mae: 1.8141 - val_loss: 3.2462 - val_mae: 3.2462
Epoch 8/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.5829 - mae: 1.5829 - val_loss: 3.2378 - val_mae: 3.2378
Epoch 9/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1.1821 - mae: 1.

### Validate Model

In [26]:
# Get prediction on the test data
y_pred_norm = model_2_3.predict(X_test_norm)

# 3. Denormalize to VND
y_pred_real = (y_pred_norm * test_ranges) + test_mins
y_test_real = (y_test_norm * test_ranges) + test_mins

for day in range(k_days):
    day_mae = mean_absolute_error(y_test_norm[:, day], y_pred_norm[:, day])
    day_mae_real = mean_absolute_error(y_test_real[:, day], y_pred_real[:, day])
    print(f"Day {day+1} -> MAE on the test set: {day_mae:.6f} | Actual Error (MAE): ${day_mae_real:.4f} USD")

478/478 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
Day 1 -> MAE on the test set: 1.238693 | Actual Error (MAE): $365.6801 USD
Day 2 -> MAE on the test set: 2.437403 | Actual Error (MAE): $508.2638 USD
Day 3 -> MAE on the test set: 3.735350 | Actual Error (MAE): $632.3850 USD


# Task 3

## Task 3.1

### Preprocess Data

In [27]:
# Define window and forecasting parameters
window_size = 30 
future_window = 7
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

def prepare_buy_features(df, future_window=7):
    df_feat = df.copy()

    # SMA
    df_feat['SMA_10'] = df_feat['Close'].rolling(window=10).mean()
    df_feat['SMA_20'] = df_feat['Close'].rolling(window=20).mean()
    
    # MACD
    df_feat['EMA_12'] = df_feat['Close'].ewm(span=12, adjust=False).mean()
    df_feat['EMA_26'] = df_feat['Close'].ewm(span=26, adjust=False).mean()
    df_feat['MACD'] = df_feat['EMA_12'] - df_feat['EMA_26']
    df_feat['Signal_Line'] = df_feat['MACD'].ewm(span=9, adjust=False).mean()
    df_feat = df_feat.drop(columns=['EMA_12', 'EMA_26'])

    # RSI
    delta = df_feat['Close'].diff()
    up = delta.clip(lower=0) 
    down = -1 * delta.clip(upper=0) 
    ema_up = up.ewm(com=13, adjust=False).mean()
    ema_down = down.ewm(com=13, adjust=False).mean()
    rs = ema_up / ema_down
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))

    # Calculate the Future Return from current to the next 7th day 
    df_feat['Future_Return'] = (df_feat['Close'].shift(-future_window) - df_feat['Close']) / df_feat['Close']
    
    # Label: buy if the Future Return larger than 1.5%
    df_feat['Buy_Signal'] = (df_feat['Future_Return'] > 0.015).astype(int)
    
    # Remove Future Return to prevent data leakage
    df_feat = df_feat.drop(columns=['Future_Return'])
    
    # SMA_20 need the first 20 days to gain momentum for calculation hence creates a NaN.
    return df_feat.dropna()

feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume', 
    'RSI_14', 'SMA_10', 'SMA_20', 'MACD', 'Signal_Line'
]
num_features = len(feature_cols)

# Split the dataset into time windows per company
for company, df in cleaned_companies_dict.items():
    df_trade = prepare_buy_features(df, future_window)
    values = df_trade[feature_cols].values.astype(np.float32)
    labels = df_trade['Buy_Signal'].values
    
    # Create sliding windows for X 
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)
    
    # Align X and y
    num_valid_samples = len(values) - window_size + 1
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = labels[window_size - 1:].reshape(-1, 1)
    
    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)
X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (48535, 30, 10)
Shape of validation set:  (12144, 30, 10)
Shape of test set:  (15183, 30, 10)


In [28]:
def normalize_windows(X, feature_names):
    X_norm = X.copy()
    
    # Declare the names of the columns that need to be normalized for each window
    cols_to_norm = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_20']
    window_norm_indices = [feature_names.index(col) for col in cols_to_norm]
    
    # Specific handling for RSI_14 (Find index by name and divide by 100)
    if 'RSI_14' in feature_names:
        rsi_index = feature_names.index('RSI_14')
        X_norm[:, :, rsi_index] = X_norm[:, :, rsi_index] / 100.0
    
    # MinMax normalization
    for i in range(len(X)):
        min_f = np.min(X[i, :, window_norm_indices], axis=0)
        max_f = np.max(X[i, :, window_norm_indices], axis=0)
        range_f = max_f - min_f
        # Prevent division by zero
        range_f[range_f == 0] = 1.0 
        
        X_norm[i, :, window_norm_indices] = (X[i, :, window_norm_indices] - min_f) / range_f
        
    return X_norm

X_train_norm = normalize_windows(X_train, feature_cols)
X_val_norm = normalize_windows(X_val, feature_cols)
X_test_norm = normalize_windows(X_test, feature_cols)

In [29]:
# Build the model architecture
model_3_1 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_3_1.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_3_1.add(BatchNormalization()) 
model_3_1.add(MaxPooling1D(pool_size=2))
model_3_1.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_3_1.add(LSTM(64, return_sequences=True)) 
model_3_1.add(Dropout(0.2))
model_3_1.add(LSTM(32, return_sequences=False))
model_3_1.add(Dropout(0.2))

# Output Block
model_3_1.add(Dense(32, activation='relu'))
model_3_1.add(Dense(1, activation='sigmoid'))

# Compile the model
metrics = [
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]
model_3_1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='binary_crossentropy', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True
)

# Train the model
class_weights = {0: 1.0, 1: 2.0}
history = model_3_1.fit(
    X_train_norm, y_train, 
    validation_data=(X_val_norm, y_val), 
    epochs=30,
    batch_size=512, 
    class_weight=class_weights,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.4191 - loss: 0.9434 - precision: 0.3686 - recall: 0.8356 - val_accuracy: 0.3874 - val_loss: 0.7171 - val_precision: 0.3142 - val_recall: 0.7943
Epoch 2/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4246 - loss: 0.9383 - precision: 0.3684 - recall: 0.8405 - val_accuracy: 0.3871 - val_loss: 0.7119 - val_precision: 0.3150 - val_recall: 0.8003
Epoch 3/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4427 - loss: 0.9372 - precision: 0.3731 - recall: 0.7983 - val_accuracy: 0.4103 - val_loss: 0.7225 - val_precision: 0.3166 - val_recall: 0.7484
Epoch 4/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4509 - loss: 0.9360 - precision: 0.3772 - recall: 0.8001 - val_accuracy: 0.3562 - val_loss: 0.7189 - val_precision: 0.3197 - val_recall: 0.9205
Epoch 5/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4576 - loss: 0.9325 - precision: 0.3789 - recall: 0.7984 - val_accuracy: 0.4396 - val_loss: 0.7156 

In [30]:
# Get predictions on the test set (Outputs are probabilities)
y_pred_prob = model_3_1.predict(X_test_norm)

# Convert probabilities to binary classes (Threshold = 0.5)
y_pred_classes = (y_pred_prob > 0.5).astype(int)

print("Accuracy on the test set: ", accuracy_score(y_test, y_pred_classes))
print("\nClassification Report:\n", classification_report(y_test, y_pred_classes))

475/475 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Accuracy on the test set:  0.40861489824145425

Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.15      0.24      9605
           1       0.37      0.85      0.51      5578

    accuracy                           0.41     15183
   macro avg       0.50      0.50      0.38     15183
weighted avg       0.54      0.41      0.34     15183



## Task 3.2

### Preprocess Data

In [31]:
# Define window and forecasting parameters
window_size = 30 
future_window = 7
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

def prepare_sell_features(df, future_window=7):
    df_feat = df.copy()

    # SMA
    df_feat['SMA_10'] = df_feat['Close'].rolling(window=10).mean()
    df_feat['SMA_20'] = df_feat['Close'].rolling(window=20).mean()
    
    # MACD
    df_feat['EMA_12'] = df_feat['Close'].ewm(span=12, adjust=False).mean()
    df_feat['EMA_26'] = df_feat['Close'].ewm(span=26, adjust=False).mean()
    df_feat['MACD'] = df_feat['EMA_12'] - df_feat['EMA_26']
    df_feat['Signal_Line'] = df_feat['MACD'].ewm(span=9, adjust=False).mean()
    df_feat = df_feat.drop(columns=['EMA_12', 'EMA_26'])

    # RSI
    delta = df_feat['Close'].diff()
    up = delta.clip(lower=0) 
    down = -1 * delta.clip(upper=0) 
    ema_up = up.ewm(com=13, adjust=False).mean()
    ema_down = down.ewm(com=13, adjust=False).mean()
    rs = ema_up / ema_down
    df_feat['RSI_14'] = 100 - (100 / (1 + rs))

    # Calculate the Future Return from current to the next 7th day 
    df_feat['Future_Return'] = (df_feat['Close'].shift(-future_window) - df_feat['Close']) / df_feat['Close']
    
    # Label: buy if the Future Return smaller than -1.5%
    df_feat['Sell_Signal'] = (df_feat['Future_Return'] < -0.015).astype(int)
    
    # Remove Future Return to prevent data leakage
    df_feat = df_feat.drop(columns=['Future_Return'])
    
    # SMA_20 need the first 20 days to gain momentum for calculation hence creates a NaN.
    return df_feat.dropna()

feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume', 
    'RSI_14', 'SMA_10', 'SMA_20', 'MACD', 'Signal_Line'
]
num_features = len(feature_cols)

# Split the dataset into time windows
for company, df in cleaned_companies_dict.items():
    df_trade = prepare_sell_features(df, future_window)
    values = df_trade[feature_cols].values.astype(np.float32)
    labels = df_trade['Sell_Signal'].values
    
    # Create sliding windows for X 
    X_company_views = sliding_window_view(values, (window_size, num_features)).reshape(-1, window_size, num_features)
    
    # Align X and y
    num_valid_samples = len(values) - window_size + 1
    X_company_final = X_company_views[:num_valid_samples]
    y_company_final = labels[window_size - 1:].reshape(-1, 1)
    
    # Split data into train, val and test. Note that 'shuffle=False' due to time-series data.
    X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
        X_company_final, y_company_final, test_size=0.2, shuffle=False
    )
    X_train_comp, X_val_comp, y_train_comp, y_val_comp = train_test_split(
        X_train_comp, y_train_comp, test_size=0.2, shuffle=False
    )
    
    # Append to lists
    X_train_list.append(X_train_comp)
    y_train_list.append(y_train_comp)
    X_val_list.append(X_val_comp)
    y_val_list.append(y_val_comp)
    X_test_list.append(X_test_comp)
    y_test_list.append(y_test_comp)

# Final concatenation
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)
X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

print("Shape of training set: ", X_train.shape)
print("Shape of validation set: ", X_val.shape)
print("Shape of test set: ", X_test.shape)

Shape of training set:  (48535, 30, 10)
Shape of validation set:  (12144, 30, 10)
Shape of test set:  (15183, 30, 10)


In [32]:
def normalize_windows(X, feature_names):
    X_norm = X.copy()
    
    # Declare the names of the columns that need to be normalized for each window
    cols_to_norm = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_20']
    window_norm_indices = [feature_names.index(col) for col in cols_to_norm]
    
    # Specific handling for RSI_14 (Find index by name and divide by 100)
    if 'RSI_14' in feature_names:
        rsi_index = feature_names.index('RSI_14')
        X_norm[:, :, rsi_index] = X_norm[:, :, rsi_index] / 100.0
    
    # MinMax normalization
    for i in range(len(X)):
        min_f = np.min(X[i, :, window_norm_indices], axis=0)
        max_f = np.max(X[i, :, window_norm_indices], axis=0)
        range_f = max_f - min_f
        # Prevent division by zero
        range_f[range_f == 0] = 1.0 
        
        X_norm[i, :, window_norm_indices] = (X[i, :, window_norm_indices] - min_f) / range_f
        
    return X_norm

X_train_norm = normalize_windows(X_train, feature_cols)
X_val_norm = normalize_windows(X_val, feature_cols)
X_test_norm = normalize_windows(X_test, feature_cols)

In [33]:
# Build the model architecture
model_3_2 = tf.keras.Sequential()

# CNN Block: Extract local patterns and short-term trends
model_3_2.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(window_size, num_features), padding='same'))
model_3_2.add(BatchNormalization()) 
model_3_2.add(MaxPooling1D(pool_size=2))
model_3_2.add(Dropout(0.2))

# LSTM Block: Capture long-term sequential dependencies
model_3_2.add(LSTM(64, return_sequences=True)) 
model_3_2.add(Dropout(0.2))
model_3_2.add(LSTM(32, return_sequences=False))
model_3_2.add(Dropout(0.2))

# Output Block
model_3_2.add(Dense(32, activation='relu'))
model_3_2.add(Dense(1, activation='sigmoid'))

# Compile the model
metrics = [
    'accuracy',
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall')
]
model_3_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss='binary_crossentropy', 
    metrics=metrics
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True
)

# Train the model
class_weights = {0: 1.0, 1: 2.0}
history = model_3_1.fit(
    X_train_norm, y_train, 
    validation_data=(X_val_norm, y_val), 
    epochs=30,
    batch_size=512, 
    class_weight=class_weights,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4837 - loss: 0.9391 - precision: 0.4011 - recall: 0.7651 - val_accuracy: 0.3944 - val_loss: 0.7359 - val_precision: 0.3479 - val_recall: 0.9290
Epoch 2/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4691 - loss: 0.9316 - precision: 0.4003 - recall: 0.8295 - val_accuracy: 0.3896 - val_loss: 0.7415 - val_precision: 0.3506 - val_recall: 0.9702
Epoch 3/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4920 - loss: 0.9253 - precision: 0.4107 - recall: 0.8219 - val_accuracy: 0.4181 - val_loss: 0.7281 - val_precision: 0.3491 - val_recall: 0.8580
Epoch 4/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4963 - loss: 0.9212 - precision: 0.4078 - recall: 0.7971 - val_accuracy: 0.4395 - val_loss: 0.7213 - val_precision: 0.3537 - val_recall: 0.8191
Epoch 5/30
95/95 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4945 - loss: 0.9250 - precision: 0.4137 - recall: 0.8084 - val_accuracy: 0.4431 - val_loss: 0.7116 

In [34]:
# Get predictions on the test set (Outputs are probabilities)
y_pred_prob = model_3_2.predict(X_test_norm)

# Convert probabilities to binary classes (Threshold = 0.5)
y_pred_classes = (y_pred_prob > 0.5).astype(int)

print("Accuracy on the test set: ", accuracy_score(y_test, y_pred_classes))
print("\nClassification Report:\n", classification_report(y_test, y_pred_classes))

475/475 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
Accuracy on the test set:  0.35599025225581243

Classification Report:
               precision    recall  f1-score   support

           0       0.57      0.07      0.13      9958
           1       0.34      0.89      0.49      5225

    accuracy                           0.36     15183
   macro avg       0.45      0.48      0.31     15183
weighted avg       0.49      0.36      0.25     15183



# Task 4

In [35]:
# List to store the prediction results for each company
portfolio_data = []

# Iterate through all companies in the cleaned data dictionary from Task 2
for company, df in cleaned_companies_dict.items():
    # Extract the last 30 trading days for this specific company
    # Assuming feature columns are (Open, High, Low, Close, Volume)
    latest_30_days = df.iloc[-window_size:, 1:6].values.astype(np.float32)
    
    # Get the current Open price
    current_open_price = latest_30_days[-1, 0] 
    
    # Normalize the data 
    X_input = latest_30_days.reshape(1, window_size, 5)
    
    # Calculate min, max, and range for normalization
    min_f = np.min(X_input[0], axis=0)
    max_f = np.max(X_input[0], axis=0)
    range_f = max_f - min_f
    range_f[range_f == 0] = 1.0 # Prevent division by zero
    
    X_input_norm = (X_input - min_f) / range_f
    
    # Predict the Open price for the kth day ahead using model_2_2
    y_pred_norm = model_2_2.predict(X_input_norm, verbose=0)
    
    # Denormalize the prediction back to VND
    predicted_open_price = (y_pred_norm[0][0] * range_f[0]) + min_f[0]
    
    # Calculate Expected Return in percentage
    expected_return_pct = round(((predicted_open_price - current_open_price) / current_open_price) * 100, 2)
    
    # Append the results to our list
    portfolio_data.append({
        'Company': company,
        'Current_Open': current_open_price,
        'Predicted_Open': predicted_open_price,
        'Expected Return (%)': expected_return_pct
    })

# Convert the list of dictionaries into a Pandas DataFrame
df_portfolio = pd.DataFrame(portfolio_data)

## Task 4.1

In [36]:
dividend_folder = '/kaggle/input/datasets/pnk1926/final-project-dataset/data-vn-20230228/dividend-history'
dividend_data_list = []

# Fetch Dividend History for each company
for company in cleaned_companies_dict.keys():
    div_files = [f for f in os.listdir(dividend_folder) if f.startswith(f"{company}-")]
    if div_files:
        div_path = os.path.join(dividend_folder, div_files[0])
        df_div = pd.read_csv(div_path)
        # Check if the dataframe has the required columns
        if not df_div.empty and 'cashYear' in df_div.columns and 'Percentage' in df_div.columns:
            # Calculate Trailing Dividend: Sum of percentages in the most recent year
            latest_year = df_div['cashYear'].max()
            recent_dividends = df_div[df_div['cashYear'] == latest_year]
            total_div_yield = recent_dividends['Percentage'].sum() * 100
            
            dividend_data_list.append({
                'Company': company,
                'Dividend Yield (%)': total_div_yield
            })

# Convert to DataFrame
df_dividends = pd.DataFrame(dividend_data_list, columns=['Company', 'Dividend Yield (%)'])

# Merge with the original df_portfolio
df_task4_1 = df_portfolio.merge(df_dividends, on='Company', how='left')

# Fill missing dividends with 0
df_task4_1['Dividend Yield (%)'] = df_task4_1['Dividend Yield (%)'].fillna(0)

# Calculate Priority Score
df_task4_1['Priority Score'] = df_task4_1['Expected Return (%)'] + df_task4_1['Dividend Yield (%)']

# Sort by Priority Score
df_task4_1 = df_task4_1.sort_values(by='Priority Score', ascending=False).reset_index(drop=True)

print("\nProfitable Stock Selection")
print(df_task4_1[['Company', 'Expected Return (%)', 'Dividend Yield (%)', 'Priority Score']].head(10))


Profitable Stock Selection
  Company  Expected Return (%)  Dividend Yield (%)  Priority Score
0     ELC                 2.75                   0            2.75
1     SAM                 1.87                   0            1.87
2     HIG                 1.69                   0            1.69
3     POT                 1.49                   0            1.49
4     SBD                 0.94                   0            0.94
5     HPT                 0.31                   0            0.31
6     CKV                 0.30                   0            0.30
7     CMG                 0.27                   0            0.27
8     FPT                 0.19                   0            0.19
9     VIE                 0.14                   0            0.14


/tmp/ipykernel_23/3956715595.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_task4_1['Dividend Yield (%)'] = df_task4_1['Dividend Yield (%)'].fillna(0)


## Task 4.2

In [37]:
financial_folder = '/kaggle/input/datasets/pnk1926/final-project-dataset/data-vn-20230228/financial-ratio'
financial_data_list = []

# Fetch Financial Ratios for each company
for company in cleaned_companies_dict.keys():
    fin_files = [f for f in os.listdir(financial_folder) if f.startswith(f"{company}-")]
    if fin_files:
        fin_path = os.path.join(financial_folder, fin_files[0])
        df_fin = pd.read_csv(fin_path)
        
        if not df_fin.empty:
            latest_fin = df_fin.iloc[0] # Get the most recent financial report
            financial_data_list.append({
                'Company': company, 
                'ROE': latest_fin.get('roe', 0),
                'P/E': latest_fin.get('priceToEarning', 0)
            })

# Convert to DataFrame
df_financials = pd.DataFrame(financial_data_list, columns=['Company', 'ROE', 'P/E'])

# Merge with the output from Task 4.1
df_task4_2 = df_task4_1.merge(df_financials, on='Company', how='left')

# Fill missing financial data with 0
df_task4_2['ROE'] = df_task4_2['ROE'].fillna(0)
df_task4_2['P/E'] = df_task4_2['P/E'].fillna(0)

# Calculate Risk Score
df_task4_2['Risk Score'] = 0

# Rule 1: AI Model predicts a price drop
df_task4_2.loc[df_task4_2['Expected Return (%)'] < 0, 'Risk Score'] += 1
# Rule 2: Company is losing money (Negative ROE)
df_task4_2.loc[df_task4_2['ROE'] < 0, 'Risk Score'] += 1
# Rule 3: Unprofitable valuation (Negative P/E)
df_task4_2.loc[df_task4_2['P/E'] < 0, 'Risk Score'] += 1

# Label the Status
df_task4_2['Risk Status'] = df_task4_2['Risk Score'].apply(
    lambda score: 'Safe' if score == 0 else 'High Risk'
)

print("\nRisk Management")
print(df_task4_2[df_task4_2['Risk Status'] == 'High Risk'][['Company', 'Expected Return (%)', 'ROE', 'P/E', 'Risk Score']].head(5))


Risk Management
   Company  Expected Return (%)    ROE   P/E  Risk Score
10     TST                 0.10 -0.080  -7.4           2
12     ICT                -0.80  0.014  43.6           1
13     LTC                -0.88 -0.071  -3.6           3
14     VTE                -1.36  0.009  70.0           1
15     PMT                -1.68  0.008  65.8           1


## Task 4.3

In [38]:
safe_df = df_task4_2[df_task4_2['Risk Score'] == 0]

# Strategy 1: Risk-Taking Portfolio
# Profile: wants maximum capital gain in the short term.
# Action: allocate equally to the best 3 stocks with the absolute highest Expected Return.
print("\nRisk-Taking Portfolio")

# Rank in descending order by Expected Return and Filter the best 3 stocks
risk_port = safe_df.nlargest(3, 'Expected Return (%)').copy()

# Equally allocate capital
risk_port['Capital Allocation (%)'] = round(100.0 / len(risk_port), 2)

# Total Expected Return
print(risk_port[['Company', 'Expected Return (%)', 'Dividend Yield (%)', 'Capital Allocation (%)']])
print(f"Expected Portfolio Total Return: {risk_port['Expected Return (%)'].mean():.2f}%")

# Strategy 2: Prudent Portfolio
# Profile: needs cash flow and strong fundamentals.
# Action: allocate equally to the best 5 stocks with highest Priority Score, sort by ROE.
print("\nPrudent Portfolio")

# Rank in descending order by Priority Score and Filter the best 5 stocks
prud_port = safe_df.nlargest(5, 'Priority Score').sort_values('ROE', ascending=False).copy()

# Equally allocate capital
prud_port['Capital Allocation (%)'] = round(100.0 / len(prud_port), 2)

# Total Expected Return
print(prud_port[['Company', 'Priority Score', 'ROE', 'Expected Return (%)', 'Dividend Yield (%)', 'Capital Allocation (%)']])
print(f"Expected Portfolio Total Profit: {prud_port['Priority Score'].mean():.2f}%")


Risk-Taking Portfolio
  Company  Expected Return (%)  Dividend Yield (%)  Capital Allocation (%)
0     ELC                 2.75                   0                   33.33
1     SAM                 1.87                   0                   33.33
2     HIG                 1.69                   0                   33.33
Expected Portfolio Total Return: 2.10%

Prudent Portfolio
  Company  Priority Score    ROE  Expected Return (%)  Dividend Yield (%)  \
3     POT            1.49  0.046                 1.49                   0   
2     HIG            1.69  0.037                 1.69                   0   
0     ELC            2.75  0.036                 2.75                   0   
1     SAM            1.87  0.001                 1.87                   0   
4     SBD            0.94  0.000                 0.94                   0   

   Capital Allocation (%)  
3                    20.0  
2                    20.0  
0                    20.0  
1                    20.0  
4               

In [39]:
model_2_1.save('model_2_1.keras')